<a href="https://colab.research.google.com/github/syedmahmoodiagents/finetuning/blob/main/Pompt_Prefix_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q peft transformers accelerate bitsandbytes --q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 10.4 MB/s eta 0:00:00


In [ ]:
# !pip install -U bitsandbytes

In [ ]:
model_name = "tiiuae/falcon-rw-1b"

In [ ]:
import os, getpass

In [ ]:
os.environ["HF_TOKEN"] = getpass.getpass("Enter your Hugging Face token: ")

Enter your Hugging Face token: ··········


In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
from transformers import Trainer, TrainingArguments

In [ ]:
from peft import get_peft_model
from peft import TaskType

In [ ]:
# these are meant for either of PromptTuning or PrefixTuning
from peft import PromptTuningConfig, PrefixTuningConfig

In [ ]:
# This is meant for LoRA only
from peft import LoraConfig

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# device_map is on auto mode meaning its automatically set the device mode for which ever is available
# tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/2.62G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

**for LoRA based Fine Tuning**

In [ ]:
config3 = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query_key_value"],  # Falcon’s attention naming
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

In [ ]:
peft_model3 = get_peft_model(model, config3)

In [ ]:
peft_model3.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    logging_dir="./logs"
)

In [1]:


trainer = Trainer(
    model=peft_model3,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator
)

trainer.train()


**Prompt Tuning**

In [ ]:
config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    prompt_tuning_init="TEXT",
    prompt_tuning_init_text="Summarize the following:",
    num_virtual_tokens=8,
    tokenizer_name_or_path=model_name,
)

In [ ]:
peft_model = get_peft_model(model, config)
peft_model.print_trainable_parameters()

trainable params: 16,384 || all params: 1,311,641,600 || trainable%: 0.0012


**Prefix Tuning**

In [ ]:
config2 = PrefixTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=8,
    prefix_projection=True,
)

In [ ]:
peft_model2 = get_peft_model(model, config2)
peft_model2.print_trainable_parameters()

trainable params: 205,637,632 || all params: 1,517,262,848 || trainable%: 13.5532


**LoRA Tuning**

In [ ]:
config3 = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query_key_value"],  # Falcon’s attention naming
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

In [ ]:
peft_model3 = get_peft_model(model, config3)
peft_model3.print_trainable_parameters()

trainable params: 1,572,864 || all params: 1,313,198,080 || trainable%: 0.1198


**Implementation of LoRA on Abirate dataset**

In [ ]:
# !pip install datasets --q

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset

In [ ]:
# Example: load tiny dataset
dataset = load_dataset("Abirate/english_quotes")

README.md: 0.00B [00:00, ?B/s]

quotes.jsonl:   0%|          | 0.00/647k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [ ]:
dt = dataset["train"].select(range(100))

In [ ]:
dt.features

{'quote': Value('string'),
 'author': Value('string'),
 'tags': List(Value('string'))}

In [ ]:
def tokenize(example):
    return tokenizer(example["quote"], truncation=True, padding="max_length", max_length=128)


In [ ]:
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'}) # Add a new pad token
    model.resize_token_embeddings(len(tokenizer)) # Resize model embeddings to include the new token
if tokenizer.pad_token is None or tokenizer.pad_token_id == -1:
    tokenizer.pad_token = tokenizer.eos_token # Fallback if no dedicated [PAD] was effectively set

In [ ]:
# dt.map(tokenize)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset({
    features: ['quote', 'author', 'tags', 'input_ids', 'attention_mask'],
    num_rows: 100
})

In [ ]:
tokenized = dt.map(tokenize)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
config3 = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query_key_value"],  # Falcon’s attention naming
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

In [ ]:
peft_model3 = get_peft_model(model, config3)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:

training_args = TrainingArguments(
    output_dir="./falcon_peft_model",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    learning_rate=5e-4,
    logging_steps=10,
    fp16=True,
    save_strategy="no",
    report_to=[],
)


In [ ]:
trainer = Trainer(
    model=peft_model3,
    args=training_args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
peft_model.save_pretrained("falcon-peft-tuned")

In [ ]:
def build_prompt(review: str) -> str:
    return f"""### Instruction:
Classify the sentiment of the review.

### Input:
{review}

### Response:
"""

In [ ]:
import torch

In [ ]:
review = "The product quality was excellent but delivery was late."

prompt = build_prompt(review)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False,
        temperature=0.0
    )

text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
prediction = text.split("Sentiment:")[-1].strip()

print("Prediction:", prediction)